# Day 34 · Agent 骨架

**配套讲义**: [`days/day-34.md`](../days/day-34.md) ｜ **需要 GPU（云机器）**

把主循环搭起来：意图理解 → 检索 → 规划 → 工具调用 → 生成回复；并加上三道护栏（最大步数 / 重复动作检测 / 成本上限），连续 10 条不出死循环。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w6.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys, torch
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.version.cuda, "| available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"gpu    : {p.name}  {p.total_memory / 1024**3:.0f} GB")
    print("bf16   :", torch.cuda.is_bf16_supported())
else:
    print("⚠️  没有 GPU —— 这一天的训练/推理跑不了。先看 docs/13-hardware-and-cost.md 租机器")

## 1. 自检三条路径

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "src.agent.agent", "--selftest"],
                   capture_output=True, text=True, cwd="..")
print(r.stdout[-3000:] or r.stderr[-3000:])

## 2. 视觉证据持久化 —— 看它怎么解决「图丢了」

In [ ]:
import sys; sys.path.insert(0, "..")
from PIL import Image, ImageDraw
import inspect
from src.agent.agent import CXAgent, AgentConfig

# 看一眼这个方法的实现思路
src = inspect.getsource(CXAgent.extract_visual_evidence)
print(src[:1400])

## 3. 护栏实验：人为制造死循环

把 max_steps 调到 20，问一个工具解决不了的问题，看它怎么绕不出来、护栏怎么拦。

In [ ]:
import sys; sys.path.insert(0, "..")
from src.agent.agent import AgentConfig

cfg = AgentConfig()
for field in ("max_steps", "max_tool_calls", "cost_limit", "repeat_threshold"):
    print(f"{field:18s} = {getattr(cfg, field, '（字段名请对照实际实现）')}")

print("\n把 max_steps 调成 20 再问『帮我预测明天的天气』——")
print("观察：模型会反复尝试不存在的工具，护栏在第 N 步拦截")

## 验收清单

- [ ] `--selftest` 三条典型 query 全部走通（查询类 / 写操作类 / 超范围类）
- [ ] **连续 10 条 query 不出死循环、不超时**（这是硬指标）
- [ ] trace JSONL 落盘，能回放每一步的工具、参数、结果、耗时
- [ ] 能演示「第二轮图片不丢」—— 第一轮传图，第二轮只发文字，模型仍能答对

**卡住了？** 回看 [`days/day-34.md`](../days/day-34.md) 第五节「容易踩的坑」。

> **明天**：`days/day-35.md` —— 端到端联调：做个能录屏的界面